# 03 — scArches query mapping

**What this does:** aligns the D6 query AnnData to the reference genes,
loads the trained scANVI as a frozen reference with new batch-adaptation
parameters, and runs the short query update (`weight_decay=0.0`, up to 100
epochs). Predictions are soft class scores; confidence is the maximum score
and entropy is also stored.

**Leakage controls:** every query label is `Unknown`; this notebook never
reads `query_eval_labels_main.csv`.

**Stop condition:** if the query contains any non-`Unknown` label or a
`cell_type` column, stop.

In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
if (REPO / "src").exists():
    sys.path.insert(0, str(REPO / "src"))
import heartmap
print("heartmap from:", Path(heartmap.__file__).parent)


In [ ]:
import anndata as ad
from heartmap.config import load_config
from heartmap.models import subset_hvg
cfg = load_config("configs/main.yaml")
query = ad.read_h5ad(cfg.query_model_input_path)
print(query.shape)
assert cfg["cell_type_key"] not in query.obs.columns
assert set(query.obs["labels_scanvi"].astype(str)) == {"Unknown"}
query_h = subset_hvg(query, cfg)  # reference gene list, zero-padding if needed
print("query aligned to", query_h.n_vars, "HVGs")


## 1. Prepare query genes and load the reference model

`prepare_query_anndata` reorders/zero-pads genes; `load_query_data` freezes reference weights. The returned parameter summary must show fewer trainable than total parameters.

In [ ]:
from heartmap.models import prepare_and_load_query
ref_dir = cfg.models_dir / f"scanvi_reference_{cfg.run_tag}"
qmodel, params = prepare_and_load_query(query_h, str(ref_dir), cfg)
print(params)
assert params["n_params_trainable"] < params["n_params_total"]


## 2. Short query update (max 100 epochs, weight_decay = 0.0)

In [ ]:
from heartmap.models import train_query_model, save_history
qmodel, qhist = train_query_model(qmodel, cfg)
save_history(qmodel, cfg.results_dir / "training" /
             f"query_mapping_history_{cfg.run_tag}.csv")


## 3. Soft predictions, confidence and entropy

Confidence = max soft score. It is a *prediction confidence score*, not a calibrated probability.

In [ ]:
from heartmap.models import predict_query
preds, scores = predict_query(qmodel, query_h, cfg)
preds.head()


In [ ]:
scores.head()


## 4. Save predictions (no truth) and the query model

In [ ]:
import pandas as pd
import anndata as ad
from heartmap import LABELS_KEY
pdir = cfg.results_dir / "predictions"
pdir.mkdir(parents=True, exist_ok=True)
preds.to_csv(pdir / f"scanvi_predictions_{cfg.run_tag}.csv", index=False)
scores.to_csv(pdir / f"scanvi_scores_{cfg.run_tag}.csv", index=False)
qmodel.save(str(cfg.models_dir / f"scanvi_query_{cfg.run_tag}"),
            overwrite=True)

# Latent for joint visualisation: predictions only, never truth.
query_h.obsm["X_scANVI"] = qmodel.get_latent_representation()
q_latent = ad.AnnData(
    X=query_h.obsm["X_scANVI"].copy(),
    obs=query_h.obs[[cfg["donor_key"], LABELS_KEY]].copy())
q_latent.obs["split"] = "query"
q_latent.obs["predicted_label"] = preds["predicted_label"].to_numpy()
q_latent.obs["confidence"] = preds["confidence"].to_numpy()
q_latent.obs["entropy"] = preds["entropy"].to_numpy()
q_latent.write_h5ad(cfg.results_dir / f"query_mapped_{cfg.run_tag}.h5ad")
print("predictions, query model and query latent saved")


Both prediction sets (baseline and scANVI) are now frozen. Only now may the sealed labels be opened — see notebook 04.